In [0]:
-- view data but only 1st 10 rows 
select * from superstore limit 10; 

 -- showing total no. of rows or data in this table
select count(*) from superstore;

select * from superstore as sales_data 
where sales is null or profit is null; -- checking is any null value in sales and profit columns

 -- checking min and max date of order from order date
select min(`Order Date`), max(`Order Date`) from superstore;
/*
finding total sales, profit and orders
*/
select sum(`Sales`) as Total_sales,
sum(`profit`) as total_profit,
count('Order ID') as total_orders 
from superstore;

/*
finding sales vs profit by categories
*/
select Category ,sum(`Sales`) as Total_sales,
sum(`Profit`) as total_profit ,
round((sum(profit)/sum(sales))*100,2) as profit_margin
from superstore
group by Category
order by Total_profit desc;

--changing column name 
alter table superstore rename column `Sub-Category` to sub_category; 

/*
Profit leakage "where comapny is losing money?"
*/
select sub_category,sum(sales) as Total_sales,
sum(Profit) as total_profit ,
round((sum(profit)/sum(sales))*100,2) as profit_margin
from superstore
group by sub_category
order by profit_margin asc;

/*Region wise performance 
which region have high sales , which have low profit
 */
select Region,
sum(Sales) as Total_sales, sum(Profit) as total_profit,
round(sum(profit)/sum(sales)*100,2) as profit_margin
from superstore
group by Region
order by profit_margin desc;

/*region + category comparison */
SELECT 
    region, Category,
    SUM(profit) AS total_profit
FROM superstore
GROUP BY region, category
ORDER BY total_profit;

/*Discount impact analysis*/
select discount,round(avg(Profit),2) as avg_profit
from superstore
group by Discount order by discount;

/*Monthly trend*/
select date_format(`Order Date`, 'MM-YYYY') as month,
sum(sales) as total_sales,
sum(profit) as total_profit
from superstore
group by month
order by year(month);

/*Changing column name */

alter table superstore rename column `Customer ID` to customer_id; 

/*identify top 10 customers with highest revenue*/
select customer_id , sum(sales) as t_spent 
from superstore
group by customer_id 
order by t_spent desc 
limit 10;

/*
customer segmentation
*/
SELECT customer_id,
    SUM(Sales) AS total_spent,
    CASE 
        WHEN SUM(Sales) > 10000 THEN 'High Value'
        WHEN SUM(Sales) BETWEEN 5000 AND 10000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS customer_segment
FROM superstore
GROUP BY customer_id
order by customer_segment ASC;

/*
Segment distribution according to segment value 
*/
SELECT 
    customer_segment,
    COUNT(*) AS total_customers
FROM (
    SELECT 
        customer_id,
        CASE 
            WHEN SUM(sales) > 10000 THEN 'High Value'
            WHEN SUM(sales) BETWEEN 5000 AND 10000 THEN 'Medium Value'
            ELSE 'Low Value'
        END AS customer_segment
    FROM superstore
    GROUP BY customer_id
) as subquery
GROUP BY customer_segment;

/*
Repeat vs one time customers 
if one time customers are high then it indicates poor retention*/
--alter table superstore rename column `Order ID` to order_id;
SELECT 
    CASE 
        WHEN total_orders = 1 THEN 'One-Time'
        ELSE 'Repeat'
    END AS customer_type,
    COUNT(*) AS total_customers
FROM (
    SELECT 
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM superstore
    GROUP BY customer_id
) subquery
GROUP BY customer_type;

/*Revenue from repeat vs one time customer pattern*/

SELECT 
    CASE 
        WHEN total_orders = 1 THEN 'One-Time'
        ELSE 'Repeat'
    END AS customer_type,
    SUM(total_spent) AS revenue
FROM (
    SELECT 
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(sales) AS total_spent
    FROM superstore
    GROUP BY customer_id
) t
GROUP BY customer_type;

/*Identifying long term customer they are key to long term revenue*/
SELECT customer_id , count(distinct order_id) as total_orders,
sum(sales) as total_spent
from superstore
group by customer_id
order by total_orders desc;